## Definição do problema de negócio

Fazer um levantamento do bolsa família atualizado em agosto de 2026, dataset no portal de transparência (https://portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia)

Confrontar com as populações das regiões, dados em https://sidra.ibge.gov.br/tabela/6579

Gerar um arquivo parquet com os dados limpos e manipulados para construção de dashboards.


## Importação dos pacotes

In [1]:
# Importa bibliotecas para manipulação dos dados
import pandas as pd
import unicodedata

## Carregando e compreendendo os dados

In [2]:
# Carrega o arquivo com os dados do bolsa familia
# Demora um pouco para carregar pois são 18.809.420 linhas
df_bolsa = pd.read_csv('202608_NovoBolsaFamilia.csv', sep=';', encoding='latin-1')

In [3]:
# Carrega o arquivo com os dados populacionais por município
# https://sidra.ibge.gov.br/tabela/6579
df_populacao = pd.read_csv('tabela6579.csv', sep=';')

In [4]:
df_bolsa.shape

(18809420, 9)

In [5]:
df_populacao.shape

(5571, 4)

In [6]:
df_bolsa.head()

,MÊS COMPETÊNCIA,MÊS REFERÊNCIA,UF,CÓDIGO MUNICÍPIO SIAFI,NOME MUNICÍPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
0,202608,202601,AL,2729,CAMPO GRANDE,NaN,1.681486e+10,MARIA CLARA NOBRE DE FRANCA,"375,00"
1,202608,202601,AL,2737,CHA PRETA,NaN,2.047755e+10,CEDIRAN DOS SANTOS SILVA,"600,00"
2,202608,202601,AL,2757,GIRAU DO PONCIANO,NaN,1.634574e+10,ICARO KAUA ALVES DOS SANTOS,"600,00"
3,202608,202601,AM,233,FONTE BOA,NaN,1.612271e+10,ALVANTINA LUZIA PEREIRA DE LIMA,"600,00"
4,202608,202601,AM,233,FONTE BOA,***.188.002-**,2.091609e+10,CARMEN FLORES MAIA,"650,00"


In [7]:
df_populacao.head()

,NÍVEL,CÓD.,MUNICÍPIO,POPULAÇÃO
0,MU,1100015,ALTA FLORESTA D'OESTE (RO),22724
1,MU,1100023,ARIQUEMES (RO),109731
2,MU,1100031,CABIXI (RO),5640
3,MU,1100049,CACOAL (RO),98884
4,MU,1100056,CEREJEIRAS (RO),16957


In [8]:
# Remover as duas primeiras colunas que são desnecessárias
df_populacao = df_populacao.iloc[:, 2:]

In [9]:
df_populacao = df_populacao.rename(columns={
    'MUNICÍPIO':    'NMMUNICIPIO',
    'POPULAÇÃO':    'QTPOPULACAO',
})

In [10]:
# Extrai o nome (tudo antes do parêntese) e a UF (as 2 letras dentro dos parênteses)
df_populacao[['NMMUNICIPIO', 'UF']] = df_populacao['NMMUNICIPIO'].str.extract(r'^(.*?)\s*\(([A-Z]{2})\)$')

In [11]:
df_populacao.head()

,NMMUNICIPIO,QTPOPULACAO,UF
0,ALTA FLORESTA D'OESTE,22724,RO
1,ARIQUEMES,109731,RO
2,CABIXI,5640,RO
3,CACOAL,98884,RO
4,CEREJEIRAS,16957,RO


In [12]:
df_populacao.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5571 entries, 0 to 5570
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   NMMUNICIPIO  5571 non-null   object
 1   QTPOPULACAO  5571 non-null   int64 
 2   UF           5571 non-null   object
dtypes: int64(1), object(2)
memory usage: 130.7+ KB


In [13]:
# Verifica linhas com valores nulos
print(df_populacao[df_populacao['QTPOPULACAO'].isna()])

Empty DataFrame
Columns: [NMMUNICIPIO, QTPOPULACAO, UF]
Index: []


In [14]:
# alterar a tipagem das colunas
df_populacao['NMMUNICIPIO'] = df_populacao['NMMUNICIPIO'].astype('string')
df_populacao['UF'] = df_populacao['UF'].astype('string')

In [15]:
# Colocar os municípios em maiúsculo caso algum não esteja e retirar as acentuações e ç
def normalizar_municipio(texto):
    if not isinstance(texto, str):
        return texto
    # Decompõe acentos (NFKD remove inclusive a cedilha)
    texto = unicodedata.normalize('NFKD', texto)
    # Remove caracteres de acentuação
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return texto.upper()

df_populacao['NMMUNICIPIO'] = df_populacao['NMMUNICIPIO'].apply(normalizar_municipio)

In [16]:
df_populacao

,NMMUNICIPIO,QTPOPULACAO,UF
0,ALTA FLORESTA D'OESTE,22724,RO
1,ARIQUEMES,109731,RO
2,CABIXI,5640,RO
3,CACOAL,98884,RO
4,CEREJEIRAS,16957,RO
...,...,...,...
5566,VIANOPOLIS,15808,GO
5567,VICENTINOPOLIS,9271,GO
5568,VILA BOA,4106,GO
5569,VILA PROPICIO,6073,GO


In [17]:
df_bf = df_bolsa.rename(columns={
    'MÊS COMPETÊNCIA':        'MESCOMPETENCIA',
    'MÊS REFERÊNCIA':         'MESREFERENCIA',
    'UF':                     'UF',
    'CÓDIGO MUNICÍPIO SIAFI': 'CDMUNSIAFI',
    'NOME MUNICÍPIO':         'NMMUNICIPIO',
    'CPF FAVORECIDO':         'CDCPFFAVORECIDO',
    'NIS FAVORECIDO':         'NIS',
    'NOME FAVORECIDO':        'NMFAVORECIDO',
    'VALOR PARCELA':          'VLPARCELA'
})

In [18]:
df_bf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18809420 entries, 0 to 18809419
Data columns (total 9 columns):
 #   Column           Dtype  
---  ------           -----  
 0   MESCOMPETENCIA   int64  
 1   MESREFERENCIA    int64  
 2   UF               object 
 3   CDMUNSIAFI       int64  
 4   NMMUNICIPIO      object 
 5   CDCPFFAVORECIDO  object 
 6   NIS              float64
 7   NMFAVORECIDO     object 
 8   VLPARCELA        object 
dtypes: float64(1), int64(3), object(5)
memory usage: 1.3+ GB


In [19]:
print(df_bf.isnull().sum())

MESCOMPETENCIA           0
MESREFERENCIA            0
UF                       0
CDMUNSIAFI               0
NMMUNICIPIO              0
CDCPFFAVORECIDO    3998980
NIS                    321
NMFAVORECIDO             0
VLPARCELA                0
dtype: int64


In [20]:
df_bf['VLPARCELA'] = df_bf['VLPARCELA'].str.replace(',', '.', regex=False).astype(float)
df_bf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18809420 entries, 0 to 18809419
Data columns (total 9 columns):
 #   Column           Dtype  
---  ------           -----  
 0   MESCOMPETENCIA   int64  
 1   MESREFERENCIA    int64  
 2   UF               object 
 3   CDMUNSIAFI       int64  
 4   NMMUNICIPIO      object 
 5   CDCPFFAVORECIDO  object 
 6   NIS              float64
 7   NMFAVORECIDO     object 
 8   VLPARCELA        float64
dtypes: float64(2), int64(3), object(4)
memory usage: 1.3+ GB


In [21]:
#Analisando o NIS que é o identificador da pessoa no bolsa família e que temo 321 sem informação
sem_nis = df_bf[df_bf['NIS'].isna()]
print(f"Total sem NIS: {len(sem_nis)}")
print(f"\nValor total dessas linhas: R$ {sem_nis['VLPARCELA'].sum():,.2f}")
print(f"Municípios afetados: {sem_nis['NMMUNICIPIO'].nunique()}")

# Amostra
print(sem_nis[['UF', 'NMMUNICIPIO', 'NIS', 'NMFAVORECIDO', 'VLPARCELA']].sample(100))

Total sem NIS: 321

Valor total dessas linhas: R$ 231,729.00
Municípios afetados: 257
          UF        NMMUNICIPIO  NIS                           NMFAVORECIDO  \
14068991  RJ     RIO DE JANEIRO  NaN  *** BENEFICIÁRIO MENOR DE 16 ANOS ***   
14748695  RN              CAICO  NaN  *** BENEFICIÁRIO MENOR DE 16 ANOS ***   
11246209  PE             GOIANA  NaN  *** BENEFICIÁRIO MENOR DE 16 ANOS ***   
17716839  SP             SANTOS  NaN  *** BENEFICIÁRIO MENOR DE 16 ANOS ***   
5830812   GO         PLANALTINA  NaN  *** BENEFICIÁRIO MENOR DE 16 ANOS ***   
...       ..                ...  ...                                    ...   
9715885   PA  NOVO REPARTIMENTO  NaN  *** BENEFICIÁRIO MENOR DE 16 ANOS ***   
13860172  RJ        NOVA IGUACU  NaN  *** BENEFICIÁRIO MENOR DE 16 ANOS ***   
9393366   PA          CASTANHAL  NaN  *** BENEFICIÁRIO MENOR DE 16 ANOS ***   
9822479   PA             PORTEL  NaN  *** BENEFICIÁRIO MENOR DE 16 ANOS ***   
5555512   GO              ARACU  NaN  *** BEN

In [22]:
# Analisando o NIS de forma geral
print("Mescompetencia:")
print(df_bf['MESCOMPETENCIA'].value_counts())
print(f"\nTotal de NIS únicos: {df_bf['NIS'].nunique():,}")
print(f"Total de linhas:     {len(df_bf):,}")

Mescompetencia:
MESCOMPETENCIA
202608    18809420
Name: count, dtype: int64

Total de NIS únicos: 18,778,328
Total de linhas:     18,809,420


In [23]:
# Verificar NIS que aparecem mais de uma vez
nis_dup = df_bf[df_bf.duplicated(subset=['NIS'], keep=False) & df_bf['NIS'].notna()]

print(f"Linhas com NIS duplicado: {len(nis_dup):,}")
print(f"NIS únicos duplicados:    {nis_dup['NIS'].nunique():,}")

# Amostra
print(nis_dup.sort_values('NIS')[['UF', 'NMMUNICIPIO', 'NIS', 'NMFAVORECIDO', 'VLPARCELA']].head(20))

Linhas com NIS duplicado: 42,883
NIS únicos duplicados:    12,112
          UF  NMMUNICIPIO           NIS                     NMFAVORECIDO  \
18134203  SP    SAO PAULO  1.011539e+10           JOSE VITORINO DA CUNHA   
19589     SP    SAO PAULO  1.011539e+10           JOSE VITORINO DA CUNHA   
30188     SP    SAO PAULO  1.011539e+10           JOSE VITORINO DA CUNHA   
7495      SP    SAO PAULO  1.011539e+10           JOSE VITORINO DA CUNHA   
12554     SP    SAO PAULO  1.011539e+10           JOSE VITORINO DA CUNHA   
17686261  SP  SANTO ANDRE  1.061625e+10        DANIEL BATISTA DOS SANTOS   
1340      SP  SANTO ANDRE  1.061625e+10        DANIEL BATISTA DOS SANTOS   
19399     SP  SANTO ANDRE  1.061625e+10        DANIEL BATISTA DOS SANTOS   
3862      SP  SANTO ANDRE  1.061625e+10        DANIEL BATISTA DOS SANTOS   
7385      SP  SANTO ANDRE  1.061625e+10        DANIEL BATISTA DOS SANTOS   
29726     SP  SANTO ANDRE  1.061625e+10        DANIEL BATISTA DOS SANTOS   
12401     SP  SANTO AN

In [24]:
# Verifica se há duplicatas exatas (todas as colunas)
duplicatas_exatas = df_bf.duplicated(keep=False).sum()
print(f"Linhas duplicadas em TODAS as colunas: {duplicatas_exatas:,}")

# Verifica se há duplicatas só pela chave (NIS)
duplicatas_nis = df_bf.duplicated(subset=['NIS'], keep=False).sum()
print(f"Linhas duplicadas só pelo NIS: {duplicatas_nis:,}")

Linhas duplicadas em TODAS as colunas: 22
Linhas duplicadas só pelo NIS: 43,204


In [25]:
# Verifica o que difere
nis_dup = df_bf[df_bf.duplicated(subset=['NIS'], keep=False) & df_bf['NIS'].notna()]
nis_top = nis_dup['NIS'].value_counts().index[0]

exemplo = df_bf[df_bf['NIS'] == nis_top]
print(exemplo.to_string())

          MESCOMPETENCIA  MESREFERENCIA  UF  CDMUNSIAFI NMMUNICIPIO CDCPFFAVORECIDO           NIS          NMFAVORECIDO  VLPARCELA
34                202608         202601  SP        6857     PIEDADE  ***.948.998-**  2.078502e+10  FLAVIO JOSE DA SILVA      600.0
1324              202608         202602  SP        6857     PIEDADE  ***.948.998-**  2.078502e+10  FLAVIO JOSE DA SILVA      600.0
3818              202608         202603  SP        6857     PIEDADE  ***.948.998-**  2.078502e+10  FLAVIO JOSE DA SILVA      600.0
7324              202608         202604  SP        6857     PIEDADE  ***.948.998-**  2.078502e+10  FLAVIO JOSE DA SILVA      600.0
12322             202608         202605  SP        6857     PIEDADE  ***.948.998-**  2.078502e+10  FLAVIO JOSE DA SILVA      600.0
19302             202608         202606  SP        6857     PIEDADE  ***.948.998-**  2.078502e+10  FLAVIO JOSE DA SILVA      600.0
29551             202608         202607  SP        6857     PIEDADE  ***.948.998-**

In [26]:
# O exemplo acima mostra que o valor é retroativo, isso pode ser visto na coluna de Mês de referência
# Então vamos considerar somente as parcelas de referencia em agosto

# Filtra só o mês corrente (sem retroativos)
df_mes_atual = df_bf[
    (df_bf['MESCOMPETENCIA'] == 202608) &
    (df_bf['MESREFERENCIA'] == 202608)
].copy()

print(f"Total geral:               {len(df_bf):,}")
print(f"Só mês corrente (202608):  {len(df_mes_atual):,}")
print(f"Diferença (retroativos):   {len(df_bf) - len(df_mes_atual):,}")

Total geral:               18,809,420
Só mês corrente (202608):  18,778,626
Diferença (retroativos):   30,794


In [27]:
# NIS duplicados APENAS no mês corrente
nis_dup_atual = df_mes_atual[
    df_mes_atual.duplicated(subset=['NIS'], keep=False) & df_mes_atual['NIS'].notna()
]

print(f"Linhas com NIS duplicado no mês corrente: {len(nis_dup_atual):,}")
print(f"NIS únicos duplicados:                    {nis_dup_atual['NIS'].nunique():,}")

# Amostra
print(nis_dup_atual.sort_values('NIS')[[
    'MESCOMPETENCIA', 'MESREFERENCIA', 'UF', 'NMMUNICIPIO', 
    'NIS', 'NMFAVORECIDO', 'VLPARCELA'
]].head(20))

Linhas com NIS duplicado no mês corrente: 0
NIS únicos duplicados:                    0
Empty DataFrame
Columns: [MESCOMPETENCIA, MESREFERENCIA, UF, NMMUNICIPIO, NIS, NMFAVORECIDO, VLPARCELA]
Index: []


In [28]:
# Como visto acima o excedente é efeito de pagamentos retroativos.
# Vamos separar então os retroativos em outro dataframe para ter somente o valor pago corrente
# e também termos esses retroativos para possível analise futura
# Correntes: mês de competência IGUAL ao mês de referência
df_corrente = df_bf[df_bf['MESCOMPETENCIA'] == df_bf['MESREFERENCIA']].copy()

# Retroativos: mês de competência DIFERENTE do mês de referência
df_retroativos = df_bf[df_bf['MESCOMPETENCIA'] != df_bf['MESREFERENCIA']].copy()

In [29]:
# Conferindo como ficou
print(f"Total geral:       {len(df_bf):,}")
print(f"Correntes:         {len(df_corrente):,}")
print(f"Retroativos:       {len(df_retroativos):,}")
print(f"Soma:              {len(df_corrente) + len(df_retroativos):,}")
print(f"Bate com o total?  {len(df_corrente) + len(df_retroativos) == len(df_bf)}")
# Os valores pagos
print("---")
print(f"Valor total geral:          R$ {df_bf['VLPARCELA'].sum():,.2f}")
print(f"Valor correntes:            R$ {df_corrente['VLPARCELA'].sum():,.2f}")
print(f"Valor retroativos:          R$ {df_retroativos['VLPARCELA'].sum():,.2f}")
print(f"Soma (corrente+retroativo): R$ {df_corrente['VLPARCELA'].sum() + df_retroativos['VLPARCELA'].sum():,.2f}")

Total geral:       18,809,420
Correntes:         18,778,626
Retroativos:       30,794
Soma:              18,809,420
Bate com o total?  True
---
Valor total geral:          R$ 12,763,249,759.00
Valor correntes:            R$ 12,742,332,559.00
Valor retroativos:          R$ 20,917,200.00
Soma (corrente+retroativo): R$ 12,763,249,759.00


In [30]:
#Verificando se existem eventuais espacos na coluna de municipio e UF
tem_espaco_mun = df_populacao['NMMUNICIPIO'].str.len() != df_populacao['NMMUNICIPIO'].str.strip().str.len()
tem_espaco_uf = df_populacao['UF'].str.len() != df_populacao['UF'].str.strip().str.len()
print(f"Linhas com espaço: {tem_espaco_mun.sum()}, {tem_espaco_uf.sum()}")

Linhas com espaço: 0, 0


In [31]:
#Ajustando as colunas para string
df_bf['NMMUNICIPIO'] = df_bf['NMMUNICIPIO'].astype('string')
df_bf['UF'] = df_bf['UF'].astype('string')
df_bf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18809420 entries, 0 to 18809419
Data columns (total 9 columns):
 #   Column           Dtype  
---  ------           -----  
 0   MESCOMPETENCIA   int64  
 1   MESREFERENCIA    int64  
 2   UF               string 
 3   CDMUNSIAFI       int64  
 4   NMMUNICIPIO      string 
 5   CDCPFFAVORECIDO  object 
 6   NIS              float64
 7   NMFAVORECIDO     object 
 8   VLPARCELA        float64
dtypes: float64(2), int64(3), object(2), string(2)
memory usage: 1.3+ GB


In [32]:
#Verificando se existem eventuais espacos na coluna de municipio e UF
tem_espaco_mun = df_bf['NMMUNICIPIO'].str.len() != df_bf['NMMUNICIPIO'].str.strip().str.len()
tem_espaco_uf = df_bf['UF'].str.len() != df_bf['UF'].str.strip().str.len()
print(f"Linhas com espaço: {tem_espaco_mun.sum()}, {tem_espaco_uf.sum()}")

Linhas com espaço: 0, 0


In [33]:
# Certificando que há somente 1 valor para competencia
print(df_bf['MESCOMPETENCIA'].unique())

[202608]


In [34]:
#Agrupando o dataframe do bolsa familia por município
df_corrente_resumo = (
    df_corrente
    .groupby(['UF', 'NMMUNICIPIO'], as_index=False)
    .agg(
        QTD_BENEFICIARIOS=('NIS', 'nunique'),   # pessoas distintas
        VALOR_TOTAL=('VLPARCELA', 'sum'),        # total pago no mês
        QTD_PARCELAS=('VLPARCELA', 'count')      # total de parcelas pagas
    )
)

print(f"Municípios únicos: {len(df_corrente_resumo):,}")
print(df_corrente_resumo.head())

Municípios únicos: 5,571
   UF   NMMUNICIPIO  QTD_BENEFICIARIOS  VALOR_TOTAL  QTD_PARCELAS
0  AC    ACRELANDIA               2039    1419010.0          2039
1  AC  ASSIS BRASIL               1694    1400005.0          1694
2  AC     BRASILEIA               4379    3112047.0          4379
3  AC        BUJARI               1923    1365655.0          1923
4  AC      CAPIXABA               1808    1289727.0          1808


In [35]:
# Agora com o dataframe de bolsa familia agrupado podemos fazer o merge com o dataframe de população
df_corrente_final = df_corrente_resumo.merge(
    df_populacao[['UF', 'NMMUNICIPIO', 'QTPOPULACAO']],
    on=['UF', 'NMMUNICIPIO'],
    how='left',
    validate='one_to_one'
)

print(f"População nula após merge: {df_corrente_final['QTPOPULACAO'].isna().sum()}")
print(df_corrente_final.head())

População nula após merge: 36
   UF   NMMUNICIPIO  QTD_BENEFICIARIOS  VALOR_TOTAL  QTD_PARCELAS  QTPOPULACAO
0  AC    ACRELANDIA               2039    1419010.0          2039      14762.0
1  AC  ASSIS BRASIL               1694    1400005.0          1694       8716.0
2  AC     BRASILEIA               4379    3112047.0          4379      28198.0
3  AC        BUJARI               1923    1365655.0          1923      14076.0
4  AC      CAPIXABA               1808    1289727.0          1808      11035.0


In [36]:
df_corrente_final.head()

,UF,NMMUNICIPIO,QTD_BENEFICIARIOS,VALOR_TOTAL,QTD_PARCELAS,QTPOPULACAO
0,AC,ACRELANDIA,2039,1419010.0,2039,14762.0
1,AC,ASSIS BRASIL,1694,1400005.0,1694,8716.0
2,AC,BRASILEIA,4379,3112047.0,4379,28198.0
3,AC,BUJARI,1923,1365655.0,1923,14076.0
4,AC,CAPIXABA,1808,1289727.0,1808,11035.0


In [38]:
# Salva o resultado como parquet para ser usado posteriormente
df_corrente_final.to_parquet("df_corrente_final.parquet", index=False)
print(f"✅ Salvo: {len(df_corrente_final):,} municípios")

# Salva também os retroativos para usar depois caso for de interesse
df_retroativos.to_parquet("df_retroativos.parquet", index=False)

✅ Salvo: 5,571 municípios
